# Quick Start: Create Your First Voronoi Cartogram

Get started with the **Voronoi cartogram** module by building an equal-area and a population cartogram of US States.

## What You'll Learn
- How to create an equal-area Voronoi cartogram with `create_voronoi_cartogram`
- How to add population weights so cell area reflects a data variable
- How to plot and label results, and read basic convergence metrics

## Prerequisites

Install **carto-flow** first:

In [ ]:
# Install prerequisites (run once in your terminal)
# pip install carto-flow

## Step 1: Import Libraries

In [ ]:
import matplotlib.pyplot as plt

import carto_flow.data as examples
import carto_flow.voronoi_cartogram as vor

## Step 2: Load Geographic Data

Load population and geometry data for the lower 48 US States.

In [ ]:
us_states = examples.load_us_census(population=True)
us_states.head()

Let's plot the original map colored by population to establish a baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
us_states.plot(
    "Population (Millions)",
    ax=ax,
    legend=True,
    cmap="RdYlGn_r",
    legend_kwds={"shrink": 0.6, "label": "Population (Millions)"},
    linewidth=0.5,
)
ax.set(title="US States — Population (original geography)")
ax.axis("off")
plt.tight_layout();

## Step 3: Create an Equal-Area Voronoi Cartogram

Calling `create_voronoi_cartogram` without `weights` runs Lloyd relaxation until every Voronoi cell has the same area. The default `RasterBackend` labels pixels on a raster grid.

In [ ]:
result_equal = vor.create_voronoi_cartogram(us_states)

## Step 4: Plot the Equal-Area Result

Each state occupies an equal-area cell. Coloring by population reveals which states are over- or under-represented in the original geography.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
result_equal.plot(
    column="Population (Millions)",
    legend=True,
    cmap="RdYlGn_r",
    legend_kwds={"shrink": 0.6, "label": "Population (Millions)"},
    ax=ax,
)
for cell, label in zip(result_equal.cells, us_states["State Abbreviation"], strict=False):
    ax.text(cell.centroid.x, cell.centroid.y, label, fontsize=7, ha="center", va="center")
ax.set(title="US States — Equal-Area Voronoi Cartogram")
ax.axis("off")
plt.tight_layout();

## Step 5: Create a Population Voronoi Cartogram

Passing `weights` makes each cell's area proportional to the corresponding column. Large-but-sparsely-populated states (Montana, Wyoming) shrink; densely-populated ones (California, Texas, New York) expand.

In [ ]:
result = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
result.plot(
    column="Population (Millions)",
    legend=True,
    cmap="RdYlGn_r",
    legend_kwds={"shrink": 0.6, "label": "Population (Millions)"},
    ax=ax,
)
for cell, label in zip(result.cells, us_states["State Abbreviation"], strict=False):
    ax.text(cell.centroid.x, cell.centroid.y, label, fontsize=7, ha="center", va="center")
ax.set(title="US States — Voronoi Cartogram (area ∝ population)")
ax.axis("off")
plt.tight_layout();

## Step 6: Read the Metrics

The `metrics` dict summarises the run. `final_area_cv` is the area coefficient of variation — lower means cells are closer to their target areas. `mean_area_error_pct` is the average percentage deviation per region.

In [ ]:
print(f"Converged:          {result.metrics['converged']}")
print(f"Iterations:         {result.metrics['n_iterations']}")
print(f"Final area CV:      {result.metrics['final_area_cv']:.4f}")
print(f"Mean area error:    {result.metrics['mean_area_error_pct']:.1f}%")

## Next Steps

- **[Choose the Right Backend](../how-to/voronoi_cartogram/choose-backend.ipynb)** — compare RasterBackend and ExactBackend: speed, accuracy, and when each is appropriate
- **[Coastal and Peninsula Regions](../how-to/voronoi_cartogram/coastal-regions.ipynb)** — use `distance_mode="geodesic"` to prevent pixel assignments across water bodies
- **[Inspect and Improve Convergence](../how-to/voronoi_cartogram/inspect-convergence.ipynb)** — read convergence curves, area errors, and displacement; tune iteration count and relaxation schedule
- **[Analyze and Fix Topology](../how-to/voronoi_cartogram/topology.ipynb)** — detect and repair satellite cells when working with grouped sub-regions
- **[Customize the Boundary](../how-to/voronoi_cartogram/boundaries.ipynb)** — change the outer clipping shape and switch between fixed, adhesive, and elastic boundary behavior

Reference documentation: [API](../../reference/voronoi_cartogram/api/) · [Options](../../reference/voronoi_cartogram/options/) · [Backends](../../reference/voronoi_cartogram/backends/) · [Result](../../reference/voronoi_cartogram/result/)